In [42]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [43]:
load_dotenv()

True

In [44]:
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

In [45]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [46]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [47]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [48]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [49]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their **quarter‑back**!',
 'explanation': '**The joke:**  \n*Why did the football team go to the bank?*  \n*Because they wanted to get their **quarter‑back**!*\n\n---\n\n### 1. The two meanings that collide  \n\n| Word / Phrase | Meaning in football | Meaning in banking |\n|---------------|--------------------|--------------------|\n| **Quarter**   | One of the four 15‑minute periods that make up a game. It’s also the term for the position “quarter‑back,” the player who receives the ball from the center and leads the offense. | A 25‑cent coin (or, more loosely, any small amount of money). |\n| **Back**      | The player who stands “back” from the line of scrimmage and throws or hands off the ball. | The opposite side of a transaction; the place where you keep money (your “back” of the account). |\n| **Quarter‑back** | The offensive leader who “backs” the play and usually throws the b

In [50]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their **quarter‑back**!', 'explanation': '**The joke:**  \n*Why did the football team go to the bank?*  \n*Because they wanted to get their **quarter‑back**!*\n\n---\n\n### 1. The two meanings that collide  \n\n| Word / Phrase | Meaning in football | Meaning in banking |\n|---------------|--------------------|--------------------|\n| **Quarter**   | One of the four 15‑minute periods that make up a game. It’s also the term for the position “quarter‑back,” the player who receives the ball from the center and leads the offense. | A 25‑cent coin (or, more loosely, any small amount of money). |\n| **Back**      | The player who stands “back” from the line of scrimmage and throws or hands off the ball. | The opposite side of a transaction; the place where you keep money (your “back” of the account). |\n| **Quarter‑back** | The offensive leader who “backs” the play and u

In [51]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why did the football team go to the bank?\n\nBecause they wanted to get their **quarter‑back**!', 'explanation': '**The joke:**  \n*Why did the football team go to the bank?*  \n*Because they wanted to get their **quarter‑back**!*\n\n---\n\n### 1. The two meanings that collide  \n\n| Word / Phrase | Meaning in football | Meaning in banking |\n|---------------|--------------------|--------------------|\n| **Quarter**   | One of the four 15‑minute periods that make up a game. It’s also the term for the position “quarter‑back,” the player who receives the ball from the center and leads the offense. | A 25‑cent coin (or, more loosely, any small amount of money). |\n| **Back**      | The player who stands “back” from the line of scrimmage and throws or hands off the ball. | The opposite side of a transaction; the place where you keep money (your “back” of the account). |\n| **Quarter‑back** | The offensive leader who “backs” the play and 

In [52]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* and they wanted to reach new heights! 🏏😂',
 'explanation': '**Explanation of the Joke**\n\n| Element | What it Means Literally | What it Means Figuratively / In Cricket |\n|---------|------------------------|------------------------------------------|\n| **“The scores were *up*”** | “Up” can describe something that is physically higher, like a ceiling or a shelf. | In sports, especially cricket, “the scores are up” means the run total is rising or the team is doing well. |\n| **“Bring a ladder”** | A ladder is a tool used to climb to a higher physical level. | The joke pretends the team thinks they need a ladder to *physically* get to the “up” scores. |\n| **“Reach new heights”** | To literally climb higher, perhaps onto a roof or a platform. | Metaphorically, “reach new heights” means achieving a better performance or a record‑breaking result. |\n\n### How th

In [53]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* and they wanted to reach new heights! 🏏😂', 'explanation': '**Explanation of the Joke**\n\n| Element | What it Means Literally | What it Means Figuratively / In Cricket |\n|---------|------------------------|------------------------------------------|\n| **“The scores were *up*”** | “Up” can describe something that is physically higher, like a ceiling or a shelf. | In sports, especially cricket, “the scores are up” means the run total is rising or the team is doing well. |\n| **“Bring a ladder”** | A ladder is a tool used to climb to a higher physical level. | The joke pretends the team thinks they need a ladder to *physically* get to the “up” scores. |\n| **“Reach new heights”** | To literally climb higher, perhaps onto a roof or a platform. | Metaphorically, “reach new heights” means achieving a better performance or a record‑breaking resu

In [54]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* and they wanted to reach new heights! 🏏😂', 'explanation': '**Explanation of the Joke**\n\n| Element | What it Means Literally | What it Means Figuratively / In Cricket |\n|---------|------------------------|------------------------------------------|\n| **“The scores were *up*”** | “Up” can describe something that is physically higher, like a ceiling or a shelf. | In sports, especially cricket, “the scores are up” means the run total is rising or the team is doing well. |\n| **“Bring a ladder”** | A ladder is a tool used to climb to a higher physical level. | The joke pretends the team thinks they need a ladder to *physically* get to the “up” scores. |\n| **“Reach new heights”** | To literally climb higher, perhaps onto a roof or a platform. | Metaphorically, “reach new heights” means achieving a better performance or a record‑breaking res

## Time Travel

Select a checkpoint from the current thread history instead of copying an ID from a previous run.
`InMemorySaver` keeps checkpoints only for the lifetime of this checkpointer instance.
After restarting the kernel or rerunning the graph setup, rerun the Cricket invocation first.
Replaying from before `generate_explanation` preserves the joke and runs the explanation again.

In [55]:
# Find the most recent checkpoint just before the explanation node.
history = list(workflow.get_state_history(config2))
replay_checkpoint = next(
    (snapshot for snapshot in history
     if snapshot.next == ("generate_explanation",)),
    None,
)
if replay_checkpoint is None:
    raise RuntimeError(
        "No replay checkpoint exists for this thread. "
        "Run the Cricket invocation cell with the current workflow first."
    )

replay_config = replay_checkpoint.config
workflow.get_state(replay_config)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* and they wanted to reach new heights! 🏏😂'}, next=('generate_explanation',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1d4545-bcba-6350-8001-76d50c519e0b'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-30T11:23:40.760920+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1d4545-b89e-6104-8000-49744ad1f20e'}}, tasks=(PregelTask(id='78baacf6-c696-3016-d0d1-6b27d9ce1975', name='generate_explanation', path=('__pregel_pull', 'generate_explanation'), error=None, interrupts=(), state=None, result={'explanation': '**Explanation of the Joke**\n\n| Element | What it Means Literally | What it Means Figuratively / In Cricket |\n|---------|------------------------|------------------------------------------|\n| **“The scores were *up*”*

In [56]:
workflow.invoke(None, config=replay_config)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the match?\n\nBecause they heard the scores were *up* and they wanted to reach new heights! 🏏😂',
 'explanation': '**What makes the joke funny?**  \n\n1. **Word‑play on “up”**  \n   - In everyday language, saying that “the scores are up” simply means the runs are **higher** than before – the team is doing well.  \n   - In the joke, “up” is taken **literally**: the players imagine the scores as something that is physically positioned above the ground, as if they could climb to it.\n\n2. **The ladder as a visual gag**  \n   - A ladder is a tool you use to reach something that’s higher than you can reach on your own. By bringing a ladder, the team is acting as if the scores are a physical object hanging in the air that they need to climb to touch.\n\n3. **“Reach new heights” – double meaning**  \n   - *Figuratively*: In sports, “reaching new heights” means improving performance, setting higher standards, or achieving

# Updating State

In [57]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d4547-4b5c-65bc-8000-f901470852eb'}}